<a href="https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane: Lane 2 — Refresh / Content Opportunity Scoring**

Mapped onto the ML loop, this is a **scoring / ranking task**, not a plain classification
task. The end output isn't a single yes/no label per page — it's an ordered list of pages,
ranked by how urgently they deserve a reviewer's attention. Underneath that ranking, the
starter pipeline uses a binary classifier (is this page declining or not) and turns its
predicted probability into a score, which is what actually gets ranked. So the task has
two layers: a classification model underneath, and a ranking/scoring problem on top, since
the real decision only cares about the top of the list, not every prediction individually.

## 2. Target or proxy

The starter pipeline's target is a **proxy label**: `is_declining_label`, defined as
`trend_direction == "down"`. This is calculated from the current 90-day window, not from a
future outcome — so it tells us how a page is behaving right now, not whether it will keep
declining or whether a fix would help. That's a real limitation worth naming honestly: this
proxy is useful for getting a working pipeline off the ground, but a stronger version of
this lane would use a genuinely future-looking label instead — e.g. features from a page's
trailing 90 days predicting whether it declines over the next 30 days. I'm keeping the
current proxy for this task-framing exercise, but flagging it as the first thing I'd
improve if I took this further.

## 3. Success metric

The right metric here is **Precision@K** (specifically Precision@50, matching what the
starter pipeline already reports), not plain accuracy. The real decision isn't "is every
prediction correct" — it's "of the top 50 pages a reviewer has time to look at, how many
are actually worth their time." A model can be mediocre across the full dataset and still
be extremely useful if it's excellent at the top of the ranking, which is the only part
anyone actually acts on. The starter pipeline's own numbers support this: baseline rules
scored Precision@50 = 0.240, while a random forest reached Precision@50 = 0.740 — a gap
that would be far less informative using overall accuracy, since accuracy would be
dominated by the huge number of pages nobody's going to review anyway.

## 4. The unit of analysis, as a real dataframe

**One row = one content page** (`content_id`), described by its trailing 90-day
performance signals (impressions, sessions, position, age, etc.) at the point of
evaluation. The dataframe below shows this directly — each row is a distinct page, not a
client, a query, or a day, which matters because the scoring/ranking output is meant to be
"here are the pages to review," one page per line.

In [ ]:
!git clone https://github.com/abbas-707/FlyRank-Internship.git
%cd FlyRank-Internship

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nOne row = one content page. Example:")
df[['content_id', 'client_id', 'impressions_90d', 'sessions_90d',
    'content_age_days', 'avg_position', 'trend_direction']].head(5)


Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 128 (delta 42), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.85 MiB | 7.69 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/FlyRank-Internship
Shape: (30000, 44)

One row = one content page. Example:


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,187,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,36.5,down
3,content_331d6c4de07b,client_19581e27de,11751,78,463,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,44.0,down


## 5. Why ML beats a fixed rule here

A fixed rule (like "flag any page with no update in 180+ days and 500+ impressions") can
only look at a couple of signals at a time, combined in a way a person guessed at. Real
pages don't fail for one clean reason — decline shows up as some mix of staleness, position
drift, thin content, and weak engagement, and the right mix differs across pages. A learned
model can weigh many signals jointly and pick up on combinations a hand-written rule would
never think to check. The starter pipeline proves this isn't just theoretical: the fixed
baseline rule reached Precision@50 = 0.240, while a random forest trained on the same
signals reached 0.740 — roughly three times as many correct picks in the top 50. That gap
is the whole argument for why this is worth an ML approach rather than another hand-tuned
rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.